# Notebook 07: FastAPI Model Serving and Docker Containerization

## Objective

In this notebook, we will convert the trained heart disease machine learning model into a web API using FastAPI.

We will:

- Understand what an API is
- Understand how machine learning model serving works
- Create a FastAPI application
- Load the saved preprocessing and prediction pipeline
- Create a health-check endpoint
- Create a prediction endpoint
- Validate input data using Pydantic
- Return the predicted class and confidence score
- Test the API using Swagger UI, browser, curl, and Postman
- Create a Dockerfile
- Build a Docker image
- Run the API inside a Docker container
- Verify that the containerized API works correctly

## What is an API?

API stands for Application Programming Interface.

An API allows two software applications to communicate with each other.

In this project:

1. A user or another application sends patient health information.
2. The FastAPI application receives the information.
3. The trained machine learning pipeline processes the information.
4. The model predicts whether the patient is at risk of heart disease.
5. The API returns the prediction and confidence score as JSON.

The API acts as a bridge between the trained machine learning model and the outside world.

## Why FastAPI is used in MLOps

FastAPI is useful for machine learning deployment because it:

- Is fast and lightweight
- Supports automatic input validation
- Automatically generates API documentation
- Works well with Python machine learning models
- Can easily be packaged inside Docker
- Provides Swagger UI for interactive testing
- Returns responses in JSON format

## Import Required Libraries

In this section, we import all the libraries needed for building our API.

These libraries help us:

- Build the FastAPI application
- Validate user input
- Load the trained machine learning model
- Perform predictions
- Return responses in JSON format

In [33]:
# ============================================================
# Import required libraries
# ============================================================

# FastAPI framework
from fastapi import FastAPI

# Pydantic is used for validating incoming JSON data
from pydantic import BaseModel

# Joblib is used to load our trained machine learning pipeline
import joblib

# NumPy helps with numerical operations
import numpy as np

# Pandas helps create DataFrames from incoming data
import pandas as pd

## Creating the FastAPI Application

The FastAPI application is the main object that manages our API.

Every API endpoint, such as `/predict` or `/health`, will be attached to this application.

When the server starts, FastAPI creates this application and waits for incoming requests.

In [34]:
# ============================================================
# Create the FastAPI application
# ============================================================

app = FastAPI(
    title="Heart Disease Prediction API",
    description="API for predicting heart disease using a trained Random Forest pipeline",
    version="1.0"
)

## Health Check Endpoint

A health check endpoint is used to verify that the API is running correctly.

Production systems, load balancers, monitoring tools, and Kubernetes frequently call this endpoint to determine whether the service is healthy and ready to receive requests.

The endpoint returns a simple JSON response indicating that the API is operational.

In [35]:
# ============================================================
# Health Check Endpoint
# ============================================================

@app.get("/health")
def health_check():
    """
    Check whether the API is running.
    """
    return {
        "status": "healthy",
        "message": "Heart Disease Prediction API is running successfully."
    }

# Create the Input Validation Model (Pydantic)

## Input Validation Using Pydantic

Before making a prediction, the API must verify that the incoming patient information is valid.

Pydantic allows us to define the expected input fields and their data types.

If any required field is missing or has an incorrect data type, FastAPI automatically returns a clear error message instead of allowing invalid data to reach the machine learning model.

In [36]:
# ============================================================
# Input Data Model
# ============================================================

class HeartDiseaseInput(BaseModel):
    age: int
    sex: int
    cp: int
    trestbps: int
    chol: int
    fbs: int
    restecg: int
    thalach: int
    exang: int
    oldpeak: float
    slope: int
    ca: float
    thal: float

# Load the Trained Machine Learning Model

## Loading the Trained Model

The machine learning pipeline created in Notebook 5 was saved using Joblib.

When the FastAPI application starts, we load this pipeline once into memory.

Keeping the model loaded improves performance because the API can reuse the same model for every prediction instead of loading it repeatedly.

In [37]:
# ============================================================
# Load the trained machine learning pipeline
# ============================================================

MODEL_PATH = "../models/heart_disease_pipeline.joblib"

model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")

Model loaded successfully.


## Create the /predict Endpoint

## Prediction Endpoint

The prediction endpoint receives patient health information in JSON format.

The input is first validated using Pydantic.

After validation, the data is converted into a Pandas DataFrame because our machine learning pipeline was trained using DataFrames.

The trained pipeline predicts:

- Heart disease class
- Prediction confidence

Finally, the API returns these results as a JSON response.

In [38]:
# ============================================================
# Prediction Endpoint
# ============================================================

@app.post("/predict")
def predict(input_data: HeartDiseaseInput):
    """
    Predict heart disease using the trained pipeline.
    """

    # Convert input data into a dictionary
    input_dict = input_data.model_dump()

    # Convert dictionary into a DataFrame
    input_df = pd.DataFrame([input_dict])

    # Make prediction
    prediction = model.predict(input_df)[0]

    # Calculate confidence score
    confidence = float(model.predict_proba(input_df)[0].max())

    # Return prediction result
    return {
        "prediction": int(prediction),
        "confidence": round(confidence, 4)
    }